In [2]:
import pandas as pd



df = pd.read_csv("../data/street_conditions_2025_labeled.csv")
print("rows and columns:", df.shape)
print(df["closed_48h"].value_counts(dropna=False))

#features
created_date = pd.to_datetime(df["created_date"])
df["hour"] = created_date.dt.hour
df["day_of_week"] = created_date.dt.day_name()
df["month"] = created_date.dt.month

#predictors and target 
features = ["descriptor", "open_data_channel_type", "hour", "day_of_week", "month"]
X = df[features].copy()
y = df["closed_48h"].copy()


print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

rows and columns: (70322, 38)
closed_48h
1    44355
0    25967
Name: count, dtype: int64
X shape: (70322, 5)
y shape: (70322,)


,descriptor,open_data_channel_type,hour,day_of_week,month
0,Pothole,UNKNOWN,0,Tuesday,4
1,Pothole,UNKNOWN,1,Tuesday,4
2,Pothole,UNKNOWN,2,Tuesday,4
3,Pothole,UNKNOWN,2,Tuesday,4
4,Pothole,UNKNOWN,4,Tuesday,4


Splitting: In the next [] cells we split the data into test and development data. Development data contains the traniing and validation set that is split later. Holding out later months for the devlopement set and leaving it for the test set 

In [3]:
test_start = pd.Timestamp("2025-11-01")


dev_mask = created_date < test_start - pd.Timedelta(hours=48)
test_mask  = created_date >= test_start

X_dev = X.loc[dev_mask].copy()
y_dev = y.loc[dev_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()


print("Development rows:", len(X_dev))
print("Test rows:", len(X_test))
print("Rows in the 48-hour gap:", (~dev_mask & ~test_mask).sum())

Development rows: 61162
Test rows: 8877
Rows in the 48-hour gap: 283


In [4]:
val_start = pd.Timestamp("2025-10-01")

dev_dates = created_date.loc[X_dev.index]
train_mask = dev_dates < val_start - pd.Timedelta(hours=48)
val_mask = dev_dates >= val_start


X_train = X_dev.loc[train_mask].copy()
y_train = y_dev.loc[train_mask].copy()
X_val = X_dev.loc[val_mask].copy()
y_val = y_dev.loc[val_mask].copy()

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Rows in the validation gap:", (~train_mask & ~val_mask).sum())


Training rows: 56157
Validation rows: 4546
Rows in the validation gap: 459


Generating a one hot encoder with column transformer since our prediction class will either be a 1 or 0

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_ftrs = [
    "descriptor",
    "open_data_channel_type",
    "hour",
    "day_of_week",
    "month",
]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_ftrs)
])

X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded validation shape:", X_val_encoded.shape)



Encoded training shape: (56157, 65)
Encoded validation shape: (4546, 65)


This is one starting model on one validation split. We’re using macro F1 as a first check and leaving the test set untouched. We haven’t tuned C or run cross-validation yet.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

model = LogisticRegression(C=1.0, max_iter=1000)
model.fit(X_train_encoded, y_train)

y_val_pred = model.predict(X_val_encoded)

val_f1 = f1_score(y_val, y_val_pred, average="macro")
print("Validation macro F1:", round(val_f1, 4))

Validation macro F1: 0.6593


This is our baseline ingorning the chosen predictors, 
We chose the majority class using training labels and scored both models on the same October validation rows using macro F1. That makes the comparison fair and keeps the November–December test set untouched.

In [7]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_encoded, y_train)

baseline_pred = baseline.predict(X_val_encoded)
baseline_f1 = f1_score(y_val, baseline_pred, average="macro")

print("Class the baseline always predicts:", baseline_pred[0])
print("Baseline validation macro F1:", round(baseline_f1, 4))
print("Logistic regression validation macro F1:", round(val_f1, 4))
print("Improvement over baseline:", round(val_f1 - baseline_f1, 4))

Class the baseline always predicts: 1
Baseline validation macro F1: 0.3764
Logistic regression validation macro F1: 0.6593
Improvement over baseline: 0.2829


Model using logistic regression was .2829 better than the baseline with using macro f1 so were scoring the classes with equal weights on both. Now lets test our model with different parameters i.e C. Higher C is equivalent to higher regularization and vice versa. 

In [8]:
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]

for C in C_values:
    candidate = LogisticRegression(C=C, max_iter=1000)
    candidate.fit(X_train_encoded, y_train)

    predictions = candidate.predict(X_val_encoded)
    score = f1_score(y_val, predictions, average="macro")

    print("C:", C, "| Validation macro F1:", round(score, 4))

C: 0.01 | Validation macro F1: 0.6579
C: 0.1 | Validation macro F1: 0.6607
C: 1.0 | Validation macro F1: 0.6593
C: 10.0 | Validation macro F1: 0.6572
C: 100.0 | Validation macro F1: 0.6577


In [9]:
validation_months = ["2025-08-01", "2025-09-01", "2025-10-01"]
folds = []

for month_start in validation_months:
    start = pd.Timestamp(month_start)
    end = start + pd.DateOffset(months=1)

    fold_train = dev_dates < start - pd.Timedelta(hours=48)
    fold_val = (dev_dates >= start) & (dev_dates < end)

    folds.append((fold_train, fold_val))

    print(start.strftime("%B"),
          "| Training rows:", fold_train.sum(),
          "| Validation rows:", fold_val.sum())

August | Training rows: 45215 | Validation rows: 5590
September | Training rows: 51024 | Validation rows: 5411
October | Training rows: 56157 | Validation rows: 4546


https://chatgpt.com/s/t_6b050165f4488191aa4b6f9bbcd0beb3 , Cloning so we have a new encoder that doesnt have the information we already learned. We take the the c values we had and compare all the c values across August September October validation sets. Help from AI to implemnt a fresh encoder with clone off of the folds I created in the prior cell. 

In [10]:
from sklearn.base import clone

scores = {C: [] for C in C_values}

for fold_train, fold_val in folds:
    X_train_fold = X_dev.loc[fold_train]
    y_train_fold = y_dev.loc[fold_train]
    X_val_fold = X_dev.loc[fold_val]
    y_val_fold = y_dev.loc[fold_val]

    # Make a fresh, unfitted copy of our preprocessing setup.
    encoder = clone(preprocessor)
    X_train_encoded_fold = encoder.fit_transform(X_train_fold)
    X_val_encoded_fold = encoder.transform(X_val_fold)

    for C in C_values:
        model = LogisticRegression(C=C, max_iter=1000)
        model.fit(X_train_encoded_fold, y_train_fold)

        predictions = model.predict(X_val_encoded_fold)
        score = f1_score(y_val_fold, predictions, average="macro")
        scores[C].append(score)

results = pd.DataFrame(scores, index=["August", "September", "October"]).T
results.index.name = "C"
results["Mean macro F1"] = results.mean(axis=1)

display(results.round(4))

,August,September,October,Mean macro F1
C,,,,
0.01,0.6567,0.6628,0.6579,0.6591
0.10,0.6679,0.6662,0.6607,0.6649
1.00,0.6669,0.6656,0.6593,0.6639
10.00,0.6609,0.6645,0.6572,0.6609
100.00,0.6592,0.6647,0.6577,0.6605


c=.1 provided the best macro score so now we will fit its score in the development data which is now January - October and we will predict the scores for Novemeber and December using this final model 

In [11]:
best_C = results["Mean macro F1"].idxmax()
print("Selected C:", best_C)

final_encoder = clone(preprocessor)
X_dev_encoded = final_encoder.fit_transform(X_dev)

logistic_model = LogisticRegression(C=best_C, max_iter=1000)
logistic_model.fit(X_dev_encoded, y_dev)

print("Training rows used for refit:", len(y_dev))

Selected C: 0.1
Training rows used for refit: 61162


READ MARKDOWN FOR TEAM: 
Logistic regression achieved a mean validation macro F1 of **0.6649**
across the August, September, and October folds. The best value we tried
was c = **0.1**. Standard deviation of **.0038**. 

-- Use the logsitc regression template as a setup for the other models so use August - Octobeer as the same valdiation set, use the chronological folds , and use the same macro f1 metric to weight the classes equally. 


Dont touch November - December until we do all of our models and see which one had the best macro F1 score 